# Machine Learning Zoomcamp

## 2. Machine Learning for Regression — Practice

Work through each exercise from memory before running it. If you get stuck, peek at the
corresponding section in the solution notebook: `02-regression.ipynb` (same folder).

Dataset: [Car price data](https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv)

Plan:

* Data preparation
* Exploratory data analysis
* Setting up the validation framework
* Linear regression (simple form)
* Linear regression (vector form)
* Training linear regression: normal equation
* Baseline model for car price prediction
* RMSE
* Validating the model
* Feature engineering
* Categorical variables
* Regularization
* Tuning the model
* Using the model


In [2]:
# Import pandas and numpy under their usual aliases
import numpy as np
import pandas as pd


## 2.2 Data preparation

1. Download the car price dataset from the URL above and read it into a DataFrame called `df`.
2. Normalize the column names: lowercase them and replace spaces with underscores.
3. Find the columns with string (`object`) dtype.
4. Normalize the *values* in those string columns too: lowercase and replace spaces with underscores.
5. Check `df.dtypes` to confirm the cleanup worked.


In [3]:
# 1-2. Download and read the dataset, normalize column names
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv'

df = pd.read_csv(data)


In [4]:
# 3. Find columns with object dtype
df.dtypes


Make                     str
Model                    str
Year                   int64
Engine Fuel Type         str
Engine HP            float64
Engine Cylinders     float64
Transmission Type        str
Driven_Wheels            str
Number of Doors      float64
Market Category          str
Vehicle Size             str
Vehicle Style            str
highway MPG            int64
city mpg               int64
Popularity             int64
MSRP                   int64
dtype: object


In [5]:
# 4. Normalize string values in those columns
df.columns = df.columns.str.lower().str.replace(" ", "_")


In [6]:
# 5. Check df.dtypes
df.dtypes
print(df['make'].dtype)


str


## 2.3 Exploratory data analysis

1. For each column, print its name, the first 5 unique values, and the number of unique values.
2. Plot the distribution of `msrp` (the price column) with a histogram (50 bins).
3. Plot the same distribution but only for `msrp < 100000` -- notice the long tail.
4. Apply `np.log1p()` to `msrp` and plot the histogram of the transformed values. Compare the shape
   to the untransformed distribution -- why do we prefer this shape for a regression target?
5. Check `df.isnull().sum()` to see which columns have missing values.

**Recall:** why do long-tail target distributions confuse ML models, and why does `log1p` (rather
than plain `log`) matter here?


In [7]:
# 1. Per-column unique value summary
for col in df.columns:
    if type(df[col][0]) == type(""):
        df[col] = df[col].str.lower().str.replace(" ", "_")
    print(col)
    print(df[col].unique())
    print(df[col].nunique())
    print()


make
<StringArray>
[          'bmw',          'audi',          'fiat', 'mercedes-benz',
      'chrysler',        'nissan',         'volvo',         'mazda',
    'mitsubishi',       'ferrari',    'alfa_romeo',        'toyota',
       'mclaren',       'maybach',       'pontiac',       'porsche',
          'saab',           'gmc',       'hyundai',      'plymouth',
         'honda',    'oldsmobile',        'suzuki',          'ford',
      'cadillac',           'kia',       'bentley',     'chevrolet',
         'dodge',   'lamborghini',       'lincoln',        'subaru',
    'volkswagen',        'spyker',         'buick',         'acura',
   'rolls-royce',      'maserati',         'lexus',  'aston_martin',
    'land_rover',         'lotus',      'infiniti',         'scion',
       'genesis',        'hummer',         'tesla',       'bugatti']
Length: 48, dtype: str
48
model
<StringArray>
[  '1_series_m',     '1_series',          '100',   '124_spider',
    '190-class',     '2_series',          

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns


In [9]:
# 2. Histogram of msrp
sns.histplot(df.msrp, bins=10)


<Axes: xlabel='msrp', ylabel='Count'>


In [10]:
# 3. Histogram of msrp < 100000
sns.histplot(df.msrp[df.msrp < 100000], bins=10)


<Axes: xlabel='msrp', ylabel='Count'>


In [11]:
# 4. log1p transform + histogram
df_msrp = np.log1p(df.msrp)
sns.histplot(df_msrp, bins=25)


<Axes: xlabel='msrp', ylabel='Count'>


In [12]:
# 5. Missing value counts
df.isnull().sum()


make                    0
model                   0
year                    0
engine_fuel_type        3
engine_hp              69
engine_cylinders       30
transmission_type       0
driven_wheels           0
number_of_doors         6
market_category      3742
vehicle_size            0
vehicle_style           0
highway_mpg             0
city_mpg                0
popularity              0
msrp                    0
dtype: int64


## 2.4 Setting up the validation framework

1. Compute `n_val`, `n_test`, `n_train` as a 20% / 20% / 60% split of the dataset (in that order --
   validation, test, train).
2. Build the row indices with `np.arange(n)`, seed the random generator (`np.random.seed(2)`) and
   shuffle the indices with `np.random.shuffle()`.
3. Use the shuffled indices to slice out `df_train`, `df_val`, `df_test`.
4. Reset the index on each of the three DataFrames (`reset_index(drop=True)`).
5. Build `y_train`, `y_val`, `y_test` as the log1p-transformed `msrp` column from each split.
6. Delete the `msrp` column from `df_train`, `df_val`, `df_test` -- why do we have to do this before
   training?

**Recall:** why do we shuffle before splitting, and why must we fix the seed?


In [13]:
# 1. Compute split sizes
print(len(df))
df = df.drop_duplicates()
print(df.duplicated().sum())
n_val = int(len(df) * 0.2)
n_test = int(len(df) * 0.2)
n_train = len(df) - n_val - n_test

print("Validation: ", n_val)
print("Testing: ", n_test)
print("Training: ", n_train)

# len(df) == n_val + n_test + n_train

# feat_cols = [col for col in df.columns if col != 'msrp']
# same_feat = df.duplicated(subset=feat_cols)
# same_feat.sum()


11914
0
Validation:  2239
Testing:  2239
Training:  6721


In [14]:
# 2. Shuffle row indices with a fixed seed
np.random.seed(12)
indexes = np.arange(len(df))
np.random.shuffle(indexes)
indexes


array([4294, 9199, 9062, ..., 3325, 9606, 5787], shape=(11199,))


In [15]:
# 3. Slice df_train / df_val / df_test using the shuffled indices
df_val = df.iloc[indexes[:n_val]]
df_train = df.iloc[indexes[n_val:n_val+n_train]]
df_test = df.iloc[indexes[n_val+n_train:]]
df_train[:1]

# df_test



In [16]:
# 4. Reset indices
df_val = df_val.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


In [53]:
# 5. Build y_train / y_val / y_test (log1p of msrp)
y_train = np.log1p(df_train.msrp)
y_test = np.log1p(df_test.msrp)
y_val = np.log1p(df_val.msrp)  


## 2.5 Linear regression (simple form)

1. Pick one row from `df_train` (e.g. `df_train.iloc[10]`) and note its feature values.
2. Write a `linear_regression(xi)` function that takes a list of feature values `xi` and returns
   `w0 + sum(w[j] * xi[j] for j in range(len(xi)))`, using some made-up `w0` and `w` (e.g. `w0 = 7.17`,
   `w = [0.01, 0.04, 0.002]`).
3. Call it on a sample `xi` and note the prediction is in log scale.
4. Use `np.expm1()` to convert the log-scale prediction back to the original price scale.

**Recall:** what does `w0` represent when all the features are zero?


In [18]:
# 1. Inspect a training row
xi = df_train.iloc[0][['engine_hp', 'city_mpg', 'popularity']].values
xi


array([np.float64(155.0), np.int64(21), np.int64(2031)], dtype=object)


In [19]:
# 2. Define linear_regression(xi) -- the loop version
w0 = 7.17
w = [0.01, 0.04, 0.002]
def linear_regression(xi):
    return w0 + sum([w[j] * xi[j] for j in range(len(xi))])


In [20]:
# 3-4. Predict on a sample xi, then undo the log1p transform
np.expm1(linear_regression(xi))


np.float64(824060.1357411505)


## 2.6 Linear regression (vector form)

1. Write a `dot(xi, w)` helper that computes the dot product of two equal-length lists by hand
   (a loop, no numpy).
2. Rewrite `linear_regression(xi)` in terms of `dot(xi, w)`.
3. Fold the bias into the weight vector: build `w_new = [w0] + w`, and rewrite `linear_regression(xi)`
   so it prepends a `1` to `xi` and calls `dot(xi, w_new)` -- no separate `w0` term needed.
4. Stack several feature rows into a matrix `X` (`np.array` of rows, each starting with a `1` for the
   bias) and rewrite `linear_regression(X)` as a single `X.dot(w_new)` call.

**Recall:** why does prepending a constant `1` to every row let us drop the separate `w0` term?


In [21]:
# 1. dot(xi, w)
def dot(xi, w):
    assert len(xi) == len(w)
    result = 0
    for j in range(len(xi)):
        result += xi[j] * w[j]
        # print(result)
    return result


In [22]:
# 2-3. linear_regression via dot(), then via the folded-in bias trick
def linear_regression(xi):
    w_new = [w0] + w
    xi_new = [1] + list(xi)
    return dot(xi_new, w_new)

print(np.expm1(linear_regression(xi)))


824060.1357411505


In [23]:
# 4. Stack rows into X and predict for all of them with one matrix multiply
np.set_printoptions(suppress=True)
X = np.ones((6721, 4))
X[:,1:] = df_train[['engine_hp', 'city_mpg', 'popularity']].values

def linear_regression(X):
    w_new = [w0] + w
    return X.dot(w_new)
np.expm1(linear_regression(X))


array([8.24060136e+05, 5.20215318e+05, 9.01665503e+05, ...,
       2.11855481e+09, 5.10935196e+05, 2.16514856e+06], shape=(6721,))


## 2.7 Training a linear regression model (normal equation)

1. Build a small feature matrix `X` (a handful of rows, a few numeric columns) and a target vector `y`.
2. Add a column of ones to `X` (the bias column) using `np.column_stack`.
3. Compute the Gram matrix `XTX = X.T.dot(X)`, invert it with `np.linalg.inv`, and compute
   `w_full = XTX_inv.dot(X.T).dot(y)` -- this is the normal equation w = (X^T X)^-1 X^T y.
4. Split `w_full` into `w0 = w_full[0]` and `w = w_full[1:]`.
5. Wrap all of the above into a `train_linear_regression(X, y)` function that returns `(w0, w)`.

**Recall:** why can't we just invert `X` directly instead of going through `X.T.dot(X)`?


In [24]:
# 1-4. Build X, y, prepend bias column, solve the normal equation by hand
# X = np.ones((10, 4))
# X[:,1:] = df_train[['engine_hp', 'city_mpg', 'popularity']].head(10).values
def train_linear_regression(X, y):
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    w0 = w_full[0]
    w = w_full[1:]
    return (w0, w)

X = df_train[['engine_hp', 'city_mpg', 'popularity']].head(10).values
X = np.column_stack([np.ones(10), X])
y = np.log1p(y_train.head(10))
train_linear_regression(X, y)


(np.float64(1.7058960665917269),
 array([ 0.00184249,  0.0158505 , -0.00000981]))


In [25]:
# 5. Wrap it into train_linear_regression(X, y)


## 2.8 Baseline model for car price prediction

1. Pick a `base` list of 5 numeric columns from `df_train` (e.g. `engine_hp`, `engine_cylinders`,
   `highway_mpg`, `city_mpg`, `popularity`).
2. Build `X_train` from those columns, filling missing values with `0` for now.
3. Train the model with `train_linear_regression(X_train, y_train)` to get `w0, w`.
4. Compute predictions `y_pred = w0 + X_train.dot(w)`.
5. Plot `y_pred` and `y_train` on the same histogram (different colors, `alpha=0.5`) to eyeball the fit.

**Recall:** why is filling missing values with `0` a simplification rather than a good default --
what would be a better filler value, and why does that also need care (train vs. val leakage)?


In [54]:
# 1-3. Build base feature list, X_train, train the model
df_train.isnull().sum()

base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 
    'city_mpg', 'popularity']

X_train = np.column_stack([np.ones(len(df_train)), df_train[base].fillna(0).values])
w0, w = train_linear_regression(X_train, y_train)
y_pred = w0 + X_train[:, 1:].dot(w)




In [55]:
# 4-5. Predict and compare y_pred vs y_train visually
sns.histplot(y_pred, bins=50, alpha=0.5, color='blue')
sns.histplot(y_train, bins=50, alpha=0.5, color='green')


<Axes: xlabel='msrp', ylabel='Count'>


## 2.9 RMSE

1. Write an `rmse(y, y_pred)` function implementing the root mean squared error:
   sqrt(mean((g(x_i) - y_i)^2)).
2. Compute the RMSE of the baseline model's predictions on the training set.

**Recall:** what does a lower RMSE mean, and why do we care about evaluating on data the model
hasn't been trained on?


In [81]:
# 1. def rmse(y, y_pred)
def rmse(y, y_pred):
    assert len(y) == len(y_pred)
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)


In [29]:
# 2. RMSE on the training predictions
rmse(y_train, y_pred)
# print(y_train, y_pred)


np.float64(0.07231885410668584)


## 2.10 Validating the model

1. Write a `prepare_X(df)` function that selects the `base` columns, fills missing values with `0`,
   and returns a numpy array -- this avoids repeating the same feature-prep logic everywhere.
2. Use `prepare_X` + `train_linear_regression` to train on `df_train`.
3. Use `prepare_X` on `df_val`, predict, and compute the validation RMSE.

**Recall:** why does wrapping feature prep in a function matter once you also need to apply it to
`df_val` and `df_test`, not just `df_train`?


In [30]:
# 2-3. Train on df_train, evaluate RMSE on df_val


In [58]:
def prepare_X(df):
    base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 
        'city_mpg', 'popularity']
    return np.column_stack([np.ones(len(df)), df[base].fillna(0).values])
    
X = prepare_X(df_train)
w0_val, w_val = train_linear_regression(X, y_train)

X_val = prepare_X(df_val)

y_val_pred = w0_val + X_val[:, 1:].dot(w_val)
no_age_rmse = rmse(y_val_pred, y_val)


In [59]:
sns.histplot(y_val_pred, bins=50, color='blue')
sns.histplot(y_val, bins=50, color='green')


<Axes: xlabel='msrp', ylabel='Count'>


## 2.11 Simple feature engineering

1. Extend `prepare_X` to add an `age` feature: `2017 - year`.
2. Retrain and re-evaluate the validation RMSE -- did it improve?
3. Plot predicted vs. actual `y_val` on the same histogram with a legend.

**Recall:** why might car age be predictive of price beyond what's already captured by the other
numeric features?


In [33]:
# 1-2. Add age to prepare_X, retrain, re-evaluate RMSE
def prepare_X(df):
    base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 
        'city_mpg', 'popularity', 'age']
     
    age = 2017 - df['year']
    df = df.copy() 
    df['age'] = age
    # print(df)
    return np.column_stack([np.ones(len(df)), df[base].fillna(0).values])


In [34]:
# 3. Plot y_pred vs y_val with a legend
X = prepare_X(df_train)
w0_val, w_val = train_linear_regression(X, y_train)

X_val = prepare_X(df_val)

y_val_pred = w0_val + X_val[:, 1:].dot(w_val)
age_rmse = rmse(y_val_pred, y_val)
age_rmse


np.float64(7.8336341540479415)


## 2.12 Categorical variables

1. Pick a handful of categorical columns (e.g. `make`, `model`, `engine_fuel_type`, `driven_wheels`,
   `market_category`, `vehicle_size`, `vehicle_style`).
2. For each one, find its top-5 most frequent values in `df_train` (`.value_counts().head()`).
3. Extend `prepare_X` to one-hot encode: for each categorical column and each of its top values,
   add a `0`/`1` column indicating whether that row has that value. Also one-hot encode
   `number_of_doors` for values `2`, `3`, `4`.
4. Retrain and check the validation RMSE -- did adding categoricals help or hurt?

**Recall:** why do we only keep the *top-5* values per category instead of one-hot encoding every
distinct value?


In [35]:
# 1-2. Pick categorical columns, find top-5 values per column in df_train
categorical_columns = ['make', 'model', 'engine_fuel_type', 'driven_wheels',
    'market_category', 'vehicle_size', 'vehicle_style']

top_5 = {}

for category in categorical_columns:
    top_5[category] = list(df[category].value_counts().head(5).index)

top_5


{'make': ['chevrolet', 'ford', 'toyota', 'volkswagen', 'nissan'],
 'model': ['silverado_1500', 'tundra', 'f-150', 'sierra_1500', 'frontier'],
 'engine_fuel_type': ['regular_unleaded',
  'premium_unleaded_(required)',
  'premium_unleaded_(recommended)',
  'flex-fuel_(unleaded/e85)',
  'diesel'],
 'driven_wheels': ['front_wheel_drive',
  'rear_wheel_drive',
  'all_wheel_drive',
  'four_wheel_drive'],
 'market_category': ['crossover',
  'flex_fuel',
  'luxury',
  'luxury,performance',
  'hatchback'],
 'vehicle_size': ['compact', 'midsize', 'large'],
 'vehicle_style': ['sedan',
  '4dr_suv',
  'coupe',
  'convertible',
  '4dr_hatchback']}


In [64]:
# 3. Extend prepare_X with one-hot encoded categoricals + num_doors
def prepare_X(df):
    df = df.copy() 
    base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 
        'city_mpg', 'popularity', 'age']

    for i in range(2,5):
        df['number_of_doors_' + str(i)] = (df['number_of_doors'] == i).astype(int)
        base.append('number_of_doors_' + str(i))
    
    for category_name in top_5:
        for entry in top_5[category_name]:
            df[category_name + "_" + entry] = (df[category_name] == entry).astype(int)
            base.append(category_name + "_" + entry)

    
            
    age = 2017 - df['year']
    
    df['age'] = age


    
    # print(df)
    return np.column_stack([np.ones(len(df)), df[base].fillna(0).values])


In [37]:
# 4. Retrain and re-check validation RMSE
prepare_X(df_train)


array([[  1., 155.,   4., ...,   1.,   0.,   0.],
       [  1.,  93.,   4., ...,   0.,   1.,   0.],
       [  1., 132.,   4., ...,   0.,   0.,   0.],
       ...,
       [  1., 255.,   8., ...,   0.,   0.,   0.],
       [  1., 402.,   8., ...,   1.,   0.,   0.],
       [  1., 260.,   6., ...,   0.,   0.,   0.]], shape=(6721, 42))


## 2.13 Regularization

1. Construct a small `X` where two columns are (near-)duplicates (one differs by `0.00000001`).
   Compute `XTX = X.T.dot(X)` and try to invert it -- what happens numerically?
2. Add a small value to the diagonal of `XTX` (`XTX + r * np.eye(...)`) before inverting -- does the
   inversion become well-behaved?
3. Write `train_linear_regression_reg(X, y, r=0.001)` -- the same normal equation as before, but with
   the diagonal regularization term added to `XTX` before inverting.
4. Retrain the full-feature model with `train_linear_regression_reg` at some fixed `r` and check the
   validation RMSE.

**Recall:** in your own words, why does adding a small constant to the diagonal fix the
near-singular matrix problem? What is this technique called?


In [38]:
# 1. Build a near-duplicate-column X, inspect XTX and try to invert it
X = np.array([
    [1, 3, 4],
    [1, 3, 4],
    [2, 3, 4]
])

XTX = X.T.dot(X)
np.linalg.inv(XTX)


LinAlgError: Singular matrix

In [38]:
# 2. Add r * np.eye(...) to the diagonal and invert again
np.linalg.inv(XTX + 0.01 * np.eye(XTX.shape[0]))


In [65]:
# 3. def train_linear_regression_reg(X, y, r=0.001)
def train_linear_regression(X, y, r=0.01):
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX + r * np.eye(XTX.shape[0]))
    w_full = XTX_inv.dot(X.T).dot(y)
    
    w0 = w_full[0]
    w = w_full[1:]
    return (w0, w)


In [80]:
# 4. Retrain with regularization and check validation RMSE
X_train = prepare_X(df_train)
X_val = prepare_X(df_val)

w0, w = train_linear_regression(X_train, y_train, 0.01)

y_train_pred = w0 + (X_train[:, 1:].dot(w))
sns.histplot(y_train_pred, color='blue')
sns.histplot(y_train, color='green')

y_val_pred = w0 + (X_val[:, 1:].dot(w))
sns.histplot(y_val_pred, color='red')
sns.histplot(y_val, color='yellow')

rmse(y_val_pred, y_val)


TypeError: 'numpy.float64' object is not callable

## 2.14 Tuning the model

1. Loop over a range of `r` values (e.g. `[0.0, 0.00001, 0.0001, 0.001, 0.1, 1, 10]`), retrain with
   each, and print `r`, `w0`, and the validation RMSE for each.
2. Pick the `r` with the best (lowest) validation RMSE and note it down.

**Recall:** why might `r=0` sometimes look fine here even though we showed a case earlier where an
un-regularized normal equation blew up?


In [85]:
# 1. Loop over candidate r values, print r / w0 / validation RMSE for each
min = None
best = None
for r in [0.0, 0.00000000001, 0.0000000001, 0.000000001, 0.00000001, 0.0000001, 0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]:
    w0, w = train_linear_regression(X_train, y_train, r=r)
    
    y_pred = w0 + (X_val[:,1:].dot(w))
    result = rmse(y_pred, y_val)

    if min == None:
        min = result
        best = r
    elif result < min:
        min = result
        best = r
    print("RMSE (%s): %s" % (str(r), str(result)))
    
min, best


RMSE (0.0): 18.646581589573387
RMSE (1e-11): 0.5112703426325753
RMSE (1e-10): 0.46391694944028106
RMSE (1e-09): 0.4623231646159003
RMSE (1e-08): 0.4622510890577133
RMSE (1e-07): 0.4622631782754665
RMSE (1e-06): 0.4622615864272953
RMSE (1e-05): 0.462261532317196
RMSE (0.0001): 0.4622615186386202
RMSE (0.001): 0.4622615436145415
RMSE (0.01): 0.4622618235159528
RMSE (0.1): 0.462267854412155
RMSE (1): 0.4624972182510941
RMSE (10): 0.4702048224730382
RMSE (100): 0.5841382639124219
RMSE (1000): 0.957306642084774
RMSE (10000): 1.2363566602677805


(np.float64(0.4622510890577133), 1e-08)


In [101]:
# 2. Note down the best r
r = 0.000000001


## 2.15 Using the model

1. Combine `df_train` and `df_val` into `df_full_train` (`pd.concat`, then reset the index) and
   likewise concatenate `y_train` and `y_val` into `y_full_train`.
2. Train the final model on the combined data with your chosen `r`.
3. Evaluate the final RMSE on the untouched `df_test` / `y_test`.
4. Pick a single row from `df_test`, wrap it in a one-row DataFrame, run it through `prepare_X`, and
   predict its price. Undo the log transform with `np.expm1()` and compare to the actual test price
   (also un-logged).

**Recall:** why do we only touch `df_test` once, right at the very end?


In [92]:
# 1. Build df_full_train / y_full_train
df_full_train = pd.concat([df_train, df_val])
df_full_train.reset_index(drop=True)

y_full_train = pd.concat([y_train, y_val])
y_full_train.reset_index(drop=True)


0        9.912497
1        7.686621
2       10.007667
3       10.254532
4       11.376567
          ...
8955     9.570529
8956    10.522207
8957    10.720974
8958    10.185126
8959    10.104222
Name: msrp, Length: 8960, dtype: float64


In [105]:
# 2-3. Train the final model, evaluate RMSE on df_test
X_full_train = prepare_X(df_full_train)
w0_final, w_final = train_linear_regression(X_full_train, y_full_train, r)

X_test = prepare_X(df_test)
y_test_pred = w0_final + (X_test[:,1:].dot(w_final))

rmse(y_test_pred, y_test)
sns.histplot(y_test_pred, color='blue', bins=50)
sns.histplot(y_test, color='green', bins=50)


<Axes: xlabel='msrp', ylabel='Count'>


In [130]:
# 4. Predict a single test row and compare to its actual (un-logged) price
test_row = df_test[98:99]
test_row = prepare_X(test_row)

prediction = np.expm1(w0_final + (test_row[:, 1:].dot(w_final)))

actual = np.expm1(y_test[98:99]).tolist()

print("Difference: " + str(abs(prediction-actual)))


Difference: [7142.81620386]


## Wrap-up

In your own words (no code), answer:

1. What problem does the validation framework solve that a single train/test split doesn't?
2. Walk through the normal equation from memory: what are the shapes of `X`, `XTX`, and `w`?
3. What's the difference between feature engineering (age, one-hot categories) and regularization --
   what problem does each one solve?
